## Imports libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import pearsonr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
import logging
import plotly.io as pio
import nbformat
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import logging
import boto3  # Simulated AWS integration
from joblib import dump
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

In [6]:
# Shows how to set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [7]:

# Simulated AWS S3 Data Loading
def load_data_from_s3(bucket, key):
    logging.info(f"Simulating data load from S3: s3://{bucket}/{key}")
    return pd.read_csv("Viral_Social_Media_Trends.csv")

In [8]:
# Data Pipeline
def process_data(df):
    logging.info("Processing data in pipeline")
    df.dropna(inplace=True)
    for col in ['Platform', 'Hashtag', 'Content_Type', 'Region', 'Engagement_Level']:
        df[col] = df[col].astype('category')
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    df['Engagement_Score'] = df[numeric_cols].mean(axis=1)
    df['Log_Views'] = np.log1p(df['Views'].clip(lower=0))
    return df, scaler

In [9]:
# LLM Features
def add_llm_features(df):
    logging.info("Adding LLM features")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['Hashtag'].tolist(), show_progress_bar=True)
    kmeans = KMeans(n_clusters=3, random_state=42)
    df['Hashtag_Cluster'] = kmeans.fit_predict(embeddings)
    cluster_names = {0: 'Trendy', 1: 'Informative', 2: 'Casual'}
    df['Cluster_Name'] = df['Hashtag_Cluster'].map(cluster_names)
    
    sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    df['Hashtag_Sentiment'] = df['Hashtag'].apply(
        lambda x: sentiment_analyzer(x)[0]['score'] if sentiment_analyzer(x)[0]['label'] == 'POSITIVE' else -sentiment_analyzer(x)[0]['score']
    )
    top_hashtags = df.groupby('Platform').apply(lambda x: x.loc[x['Engagement_Score'].idxmax(), 'Hashtag']).to_dict()
    df['Top_Hashtag_Platform'] = df['Platform'].map(top_hashtags)
    return df

In [10]:
# Predictive Modeling
def train_predictive_model(df):
    logging.info("Training Random Forest model for Engagement_Score prediction")
    features = ['Views', 'Likes', 'Shares', 'Comments', 'Hashtag_Sentiment', 'Hashtag_Cluster']
    X = df[features]
    y = df['Engagement_Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    logging.info(f"Model Performance - MSE: {mse:.4f}, R2: {r2:.4f}")
    
    # Simulate saving to SageMaker/S3
    dump(rf_model, 'rf_model.joblib')
    logging.info("Model saved locally as 'rf_model.joblib' (simulating S3 upload)")
    return rf_model, mse, r2

In [14]:
# Visualization Generation
def generate_visualizations(df):
    logging.info("Generating visualizations")
    # Bar Chart
    platform_eng = df.groupby('Platform').agg({'Engagement_Score': 'mean'}).reset_index()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=platform_eng, x='Platform', y='Engagement_Score', palette='colorblind')
    plt.title("Average Engagement by Platform", fontsize=16)
    plt.savefig("engagement_by_platform.png", dpi=300)
    plt.close()

In [20]:
# Main Execution
logging.info("Starting EDA pipeline")
bucket, key = "my-social-media-bucket", "Viral_Social_Media_Trends.csv"
df = load_data_from_s3(bucket, key)
df, scaler = process_data(df)
df = add_llm_features(df)
rf_model, mse, r2 = train_predictive_model(df)
generate_visualizations(df)

2025-03-18 07:36:45,791 - INFO - Starting EDA pipeline
2025-03-18 07:36:45,792 - INFO - Simulating data load from S3: s3://my-social-media-bucket/Viral_Social_Media_Trends.csv
2025-03-18 07:36:45,819 - INFO - Processing data in pipeline
2025-03-18 07:36:45,840 - INFO - Adding LLM features
2025-03-18 07:36:45,890 - INFO - Use pytorch device_name: mps
2025-03-18 07:36:45,890 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/phionanamugga/Documents/coding/datascience/EDA_Viral_Social_Media_Posts/.venv/lib/python3.9/site-packages/joblib/externals/loky/backend/context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[Errno 20] Not a directory: 'sysctl'
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/Users/phionanamugga/Documents/coding/datascience/EDA_Viral_Social_Media_Posts/.venv/lib/python3.9/site-packages/joblib/externals/loky/backend/context.py", line 270, in _count_physical_cores
    cpu_info = subprocess.run(
  Fi

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use mps:0
/var/folders/_f/952nnrtj2cqfx6j55v126fl80000gn/T/ipykernel_1225/2757030768.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top_hashtags = df.groupby('Platform').apply(lambda x: x.loc[x['Engagement_Score'].idxmax(), 'Hashtag']).to_dict()
/var/folders/_f/952nnrtj2cqfx6j55v126fl80000gn/T/ipykernel_1225/2757030768.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_hashtags = df.groupby('Platform').apply(lambda x: x.loc[x['Engagement_Score'].idxmax(), 'Hashtag']).to_dict()
2025-

In [21]:
# Summary for Stakeholders
logging.info("Generating Summary")
top_platform = df.groupby('Platform')['Engagement_Score'].mean().idxmax()
top_cluster = df.groupby('Cluster_Name')['Engagement_Score'].mean().idxmax()
content_dist = df['Content_Type'].value_counts()
pivot = df.pivot_table(values='Engagement_Score', index='Region', columns='Cluster_Name', aggfunc='mean')
summary_text = f"""
Key Insights for Stakeholders:
- Top Platform: {top_platform} (highest engagement)
- Top Hashtag Cluster: {top_cluster} (drives trends)
- Content Focus: {content_dist.idxmax()} dominates ({content_dist.max()} posts)
- Sentiment Note: Positive sentiment linked to higher engagement (BERT-driven)
- Predictive Model: Random Forest R2 = {r2:.2f}, MSE = {mse:.4f}
- Action: Prioritize {top_platform} in {pivot.idxmax()[top_cluster]} with trending content
- AWS Scalability: Ready for S3 storage and SageMaker deployment
"""

2025-03-18 07:57:33,518 - INFO - Generating Summary
/var/folders/_f/952nnrtj2cqfx6j55v126fl80000gn/T/ipykernel_1225/1625378109.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top_platform = df.groupby('Platform')['Engagement_Score'].mean().idxmax()
/var/folders/_f/952nnrtj2cqfx6j55v126fl80000gn/T/ipykernel_1225/1625378109.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(values='Engagement_Score', index='Region', columns='Cluster_Name', aggfunc='mean')


In [23]:
# Save Processed Data
df.to_pickle("processed_data_amazon.pkl")
logging.info("EDA core completed and data saved")


2025-03-18 07:59:07,188 - INFO - EDA core completed and data saved
